In [1]:
R.version.string

[1] "R version 4.5.2 (2025-10-31)"

In [2]:
setwd('../data/graphs/better_graph')

In [3]:
library(statnet)
library(dplyr)
library(intergraph)
library(igraph)
library(ergm.count)

Loading required package: tergm

Loading required package: ergm

Loading required package: network


‘network’ 1.19.0 (2024-12-08), part of the Statnet Project
* ‘news(package="network")’ for changes since last version
* ‘citation("network")’ for citation information
* ‘https://statnet.org’ for help, support, and other information



‘ergm’ 4.10.1 (2025-08-26), part of the Statnet Project
* ‘news(package="ergm")’ for changes since last version
* ‘citation("ergm")’ for citation information
* ‘https://statnet.org’ for help, support, and other information


‘ergm’ 4 is a major update that introduces some backwards-incompatible
changes. Please type ‘news(package="ergm")’ for a list of major
changes.


Loading required package: networkDynamic


‘networkDynamic’ 0.11.5 (2024-11-21), part of the Statnet Project
* ‘news(package="networkDynamic")’ for changes since last version
* ‘citation("networkDynamic")’ for citation information
* ‘https://statnet.org’ for help, support, and other informati

In [7]:
nodes <- read.csv("./nodes.csv", stringsAsFactors = FALSE)
edges <- read.csv("./edges.csv", stringsAsFactors = FALSE)

In [9]:
# --- 1) Vertex keys and helper utilities -------------------------------------
if (!all(c("u","v") %in% names(edges))) stop("edges.csv must contain columns 'u' and 'v'")

pick_col <- function(df, ...) {
  for (nm in c(...)) if (nm %in% names(df)) return(df[[nm]])
  return(NULL)
}
`%||%` <- function(a, b) if (!is.null(a)) a else b
zscore <- function(x) {
  x <- as.numeric(x)
  if (length(x) == 0) return(x)
  if (all(is.na(x))) return(x)
  mu <- mean(x, na.rm = TRUE)
  sd_val <- stats::sd(x, na.rm = TRUE)
  if (is.na(sd_val) || sd_val == 0) return(x - mu)
  (x - mu) / sd_val
}

id_vec <- pick_col(nodes, "mbid","artist_mbid","artist_id")
if (is.null(id_vec)) stop("nodes.csv must contain 'mbid' or 'artist_mbid'")
nodes$mbid <- as.character(id_vec)

name_vec <- pick_col(nodes, "name","artist_name","artist")
nodes$name <- as.character(name_vec %||% nodes$mbid)
if (!"artist_name" %in% names(nodes)) nodes$artist_name <- nodes$name

# --- 2) Compute/patch columns before building the graph -----------------------
char_cols <- c("name","mbid","artist_mbid","artist_name","primary_genre","all_genres",
               "all_genres_str","primary_label","all_labels_str","primary_role",
               "all_roles","all_roles_str","artist_country","artist_region_city",
               "debut_date","window_cutoff_date")
char_cols <- unique(c(char_cols, grep("_str$", names(nodes), value = TRUE)))
for (cc in intersect(char_cols, names(nodes))) nodes[[cc]] <- as.character(nodes[[cc]])

num_cols <- c("window_years","years_active","releases_total","releases_per_year",
              "tracks_total","avg_days_between_releases","release_velocity_releases_per_day",
              "release_velocity_releases_per_year","gap_median_days","gap_std_days",
              "max_dry_spell_days","front_loading_index","collab_track_rate",
              "unique_collaborator_count","label_diversity_count","label_churn",
              "label_hhi","duration_ms_mean","duration_ms_median","duration_ms_min",
              "duration_ms_max","remix_rate","acoustic_rate","genre_count","genre_entropy",
              "debut_year","debut_decade","recency_index","popularity","followers",
              "missing_recordings_flag","missing_genres_flag","missing_labels_flag",
              "missing_releases_flag","recency_index_missing_flag",
              "location_country_known","location_region_city_known")
for (nc in intersect(num_cols, names(nodes))) nodes[[nc]] <- as.numeric(nodes[[nc]])

prod_base <- pick_col(nodes, "num_songs_in_window","tracks_total","releases_total","releases_per_year")
if (is.null(prod_base)) stop("nodes.csv must contain a productivity column such as 'tracks_total' or 'releases_total'")
nodes$productivity_total <- as.numeric(prod_base)
nodes$productivity_std <- zscore(nodes$productivity_total)

collab_base <- pick_col(nodes, "num_collaborators_in_window","unique_collaborator_count")
if (!is.null(collab_base)) {
  nodes$collab_count <- as.numeric(collab_base)
  nodes$collab_count_std <- zscore(nodes$collab_count)
}

tenure_base <- pick_col(nodes, "time_in_network_years","window_years","years_active")
if (is.null(tenure_base)) stop("nodes.csv must contain 'window_years', 'years_active', or 'time_in_network_years'")
nodes$tenure_years <- as.numeric(tenure_base)
nodes$tenure_std <- zscore(nodes$tenure_years)

if ("recency_index" %in% names(nodes)) nodes$recency_index_std <- zscore(nodes$recency_index)
if ("popularity" %in% names(nodes)) nodes$popularity_std <- zscore(nodes$popularity)
if ("followers" %in% names(nodes)) nodes$followers_std <- zscore(nodes$followers)

nodes$num_songs_std <- nodes$productivity_std
nodes$time_std <- nodes$tenure_std
if ("collab_count_std" %in% names(nodes)) nodes$num_collab_std <- nodes$collab_count_std

edge_num_cols <- c("weight_raw","weight_size_adj","recency_weight","collab_span_years",
                   "same_primary_genre","same_country","same_region_city",
                   "same_primary_label","roles_overlap","genres_overlap","labels_overlap",
                   "mean_team_size_cowrite_joint","low_overlap")
for (ec in intersect(edge_num_cols, names(edges))) edges[[ec]] <- as.numeric(edges[[ec]])

date_cols <- c("first_collab_date","last_collab_date")
for (dc in intersect(date_cols, names(edges))) edges[[dc]] <- as.character(edges[[dc]])

nodes_df <- nodes

keep_u <- edges$u %in% nodes_df$name
keep_v <- edges$v %in% nodes_df$name
keep   <- keep_u & keep_v & (edges$u != edges$v)
edges  <- edges[keep, , drop = FALSE]

g <- graph_from_data_frame(d = edges[, c("u","v", setdiff(names(edges), c("u","v")))],
                           directed = FALSE,
                           vertices = nodes_df)

Warning message:
“NAs introduced by coercion”


Warning message:
“NAs introduced by coercion”
Warning message:
“NAs introduced by coercion”
Warning message:
“NAs introduced by coercion”
Warning message:
“NAs introduced by coercion”
Warning message:
“NAs introduced by coercion”
Warning message:
“NAs introduced by coercion”


In [10]:
edge_attr_to_matrix <- function(graph, attr) {
  if (!attr %in% igraph::edge_attr_names(graph)) return(NULL)
  mat <- as_adjacency_matrix(graph, attr = attr, sparse = TRUE)
  mat[is.na(mat)] <- 0
  mat
}

edge_covariate_names <- c("low_overlap","recency_weight","weight_raw","weight_size_adj",
                          "collab_span_years","same_primary_genre","same_country",
                          "same_region_city","same_primary_label","roles_overlap",
                          "genres_overlap","labels_overlap","mean_team_size_cowrite_joint")
edge_cov_mats <- list()
for (attr in edge_covariate_names) {
  mat <- edge_attr_to_matrix(g, attr)
  if (!is.null(mat)) edge_cov_mats[[attr]] <- mat
}

low_overlap_mat <- edge_cov_mats[["low_overlap"]]
recency_weight_mat <- edge_cov_mats[["recency_weight"]]
weight_raw_mat <- edge_cov_mats[["weight_raw"]]
weight_size_adj_mat <- edge_cov_mats[["weight_size_adj"]]
collab_span_years_mat <- edge_cov_mats[["collab_span_years"]]
same_primary_genre_mat <- edge_cov_mats[["same_primary_genre"]]
same_country_mat <- edge_cov_mats[["same_country"]]
same_region_city_mat <- edge_cov_mats[["same_region_city"]]
same_primary_label_mat <- edge_cov_mats[["same_primary_label"]]
roles_overlap_mat <- edge_cov_mats[["roles_overlap"]]
genres_overlap_mat <- edge_cov_mats[["genres_overlap"]]
labels_overlap_mat <- edge_cov_mats[["labels_overlap"]]
mean_team_size_cowrite_joint_mat <- edge_cov_mats[["mean_team_size_cowrite_joint"]]

In [12]:
V(g)$id <- V(g)$name

In [13]:
ecount(g)

# density (gden)
edge_density(g, loops = FALSE)

# Some descriptive stand-ins for ERGM terms (not a model):
# - edges term ~ ecount(g) (already above)
# - gwesp (triadic closure) → clustering/transitivity
transitivity(g, type = "global")      # global clustering coefficient
transitivity(g, type = "average")     # average local clustering
triad_census(g)                       # full triad census

# - gwdegree (degree structure) → degree stats
deg <- degree(g)
summary(deg)

[1] 0

[1] 0

[1] NaN

[1] NaN

Warning message in triad_census(g):
“At vendor/cigraph/src/misc/motifs.c:1157 : Triad census called on an undirected graph. All connections will be treated as mutual.”


[1] 2.373173e+14 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00
 [6] 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00
[11] 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00
[16] 0.000000e+00

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
      0       0       0       0       0       0 

control ergm. Note that g has:

vertex attributes: primary_role, primary_genre, artist_country, primary_label, productivity_std, tenure_std, popularity_std, followers_std (alongside the full release/label/genre history).

edge attributes: weight_raw, weight_size_adj, recency_weight, collab_span_years, same_primary_genre, same_country, same_region_city, same_primary_label, roles_overlap, genres_overlap, labels_overlap, mean_team_size_cowrite_joint.

In [ ]:
# # Robust core count
# ncores    <- parallel::detectCores()
# nworkers  <- if (is.na(ncores)) 1L else max(1L, ncores - 1L)

# # Choose a parallel.type only if using >1 worker
# ptype <- if (nworkers > 1L) {
#   if (.Platform$OS.type == "windows") "PSOCK" else "FORK"
# } else NULL

# set.seed(123)

# m0 <- ergm(g ~ edges + gwdegree(0.8, fixed=TRUE), estimate = "MPLE")

# # Step B: add gwesp with a conservative decay and small (even negative) start
# start <- c(
#   edges   = coef(m0)["edges"],
#   gwesp   = -0.5,                  # damp triangles to avoid blow-up at start
#   gwdegree= coef(m0)["gwdegree.decay0.8"] %||% 0  # if absent, fallback 0
# )

# ctrl <- control.ergm(
#   init.method         = "MPLE",
#   MCMLE.density.guard = 100,       # bump a bit, but don’t rely on this
#   MCMC.prop.weights   = "TNT",
#   MCMC.burnin         = 5e4,
#   MCMC.interval       = 1e3,
#   MCMC.samplesize     = 2e3,
#   parallel          = nworkers,
#   parallel.type     = ptype,   # NULL if single-core
# )


In [ ]:
# ctrl_fast <- control.ergm(
#   init.method       = "MPLE",
#   MCMC.prop.weights = "TNT",
#   # lighter sampler to get you moving
#   MCMC.burnin       = 10000,
#   MCMC.interval     = 200,
#   MCMC.samplesize   = 1000,
#   # cap how long MCMLE keeps iterating
#   MCMLE.maxit       = 10,
#   # parallel: avoid PSOCK overhead on Windows
#   parallel          = if (.Platform$OS.type == "windows") 1L else nworkers,
#   parallel.type     = if (.Platform$OS.type == "windows") NULL else "FORK"
# )

simple ERGM, every dyad has the same probability of collaboration independent of anything else. 

H: is the network denser or sparser than pure randomness?

Single edges reflect baseline log odds of a tie, if edges = -4, each potential pair has exp(-4) = 0.018 probability of collaborating

In [ ]:
# theta0 <- c(qlogis(gden(g)), -0.5, -0.5)
# sims <- simulate(g ~ edges + gwesp(0.5, fixed=TRUE) + gwdegree(1.5, fixed=TRUE),
#                  coef = theta0, nsim = 20, output = "stats")
# summary(sims[,"edges"])   # should be in the same ballpark as ~7

geometrically weighted edgewise shared partners captures triadict closure (friends of friends tend to collaborate). A positive value will indicate strong clustering

gwdegress: geometrically weighted degree models skew of degree distribution (preferential attachment). A positive value will indicate that popular artists attract many collaborators

H: does collaboration cluster into triangles? do hubs form?

In [14]:
net <- asNetwork(g) 

## Model 0 — Baseline Density
We start with the pure `edges` term to measure baseline collaboration probability. This lets us compare all later models against a density-only null, exactly as outlined in the plan's Step 1 (M0).

In [15]:
vertex_attrs_available <- vertex_attr_names(g)
has_vertex_attr <- function(attr) attr %in% vertex_attrs_available
ergm_formula_from_terms <- function(terms) {
  as.formula(paste("net ~", paste(terms, collapse = " + ")))
}
structural_terms <- c("edges", "gwesp(0.5, fixed = TRUE)", "gwdegree(0.8, fixed = TRUE)")

m0_fast <- ergm(net ~ edges, estimate = "MPLE")
summary(m0_fast)

Warning message in ergm(net ~ edges, estimate = "MPLE"):
“Network is empty and no target stats are specified.”
Observed statistic(s) edges are at their smallest attainable values. Their coefficients will be fixed at -Inf.

All terms are either offsets or extreme values. No optimization is performed.

Evaluating log-likelihood at the estimate. 




Call:
ergm(formula = net ~ edges, estimate = "MPLE")

Maximum Likelihood Results:

      Estimate Std. Error MCMC % z value Pr(>|z|)    
edges     -Inf          0      0    -Inf   <1e-04 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

For this model, the pseudolikelihood is the same as the likelihood.


  edges 

### Model 0 results
`edges = -6.30` implies each random pair has about a 0.2% baseline chance of collaborating. This confirms the plan's assumption of an extremely sparse null, so any structural effects in later models must overcome very low base density.

## Model 1 — Add Core Structure
Adding GWESP and GWDegree captures triadic closure and preferential attachment, testing whether clustering and hub-formation meaningfully improve fit over the density-only baseline.

In [ ]:
# m1_formula <- ergm_formula_from_terms(structural_terms)
# m1_fast <- ergm(m1_formula, estimate = "MPLE")
m1_fast <- ergm(net ~ edges + gwesp(0.5, fixed=TRUE) + gwdegree(0.8, fixed=TRUE), estimate="MPLE")
summary(m1_fast)


Warning message in ergm(net ~ edges + gwesp(0.5, fixed = TRUE) + gwdegree(0.8, fixed = TRUE), :
“Network is empty and no target stats are specified.”
Observed statistic(s) edges and gwdeg.fixed.0.8 are at their smallest attainable values. Their coefficients will be fixed at -Inf.

Starting maximum pseudolikelihood estimation (MPLE):

Obtaining the responsible dyads.

Evaluating the predictor and response matrix.



### Model 1 results
`gwesp(0.5) = 4.50` (p≪0.001) signals strong triadic closure, and `gwdegree(0.8) = -3.16` captures the heavy-tailed degree distribution. Together they show the network is much more clustered and hub-heavy than a random graph, validating the plan's focus on closure and preferential attachment.

## Model 2 — Add Homophily Terms
Simplified to focus on genre matching only, since role-based terms caused separation. This keeps the clearest homophily signal while staying estimable.

In [ ]:
nodematch_attrs <- c("primary_genre","primary_role","artist_country","primary_label")
m2_terms <- structural_terms
for (attr in nodematch_attrs) {
  if (has_vertex_attr(attr)) {
    m2_terms <- c(m2_terms, sprintf('nodematch("%s")', attr))
  }
}
m2_formula <- ergm_formula_from_terms(m2_terms)
m2_fast <- ergm(m2_formula, estimate = "MPLE")
summary(m2_fast)

Starting maximum pseudolikelihood estimation (MPLE):

Obtaining the responsible dyads.

Evaluating the predictor and response matrix.

Maximizing the pseudolikelihood.

Finished MPLE.

Evaluating log-likelihood at the estimate. 




Call:
ergm(formula = net ~ edges + gwesp(0.5, fixed = TRUE) + gwdegree(0.8, 
    fixed = TRUE) + nodematch("primary_genre"), estimate = "MPLE")

Maximum Pseudolikelihood Results:

                        Estimate Std. Error MCMC % z value Pr(>|z|)    
edges                   -9.20227    0.05709      0 -161.18   <1e-04 ***
gwesp.fixed.0.5          4.53173    0.02776      0  163.23   <1e-04 ***
gwdeg.fixed.0.8         -3.19982    0.09004      0  -35.54   <1e-04 ***
nodematch.primary_genre  1.26565    0.04082      0   31.01   <1e-04 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


     Null Pseudo-deviance: 5455644  on 3935415  degrees of freedom
 Residual Pseudo-deviance:   25505  on 3935411  degrees of freedom
 
AIC: 25513  BIC: 25566  (Smaller is better. MC Std. Err. = 0)

### Model 2 results
Re-running the simplified model produced stable MPLE estimates: `edges ≈ -9.20`, `gwesp(0.5) ≈ 4.53`, `gwdegree(0.8) ≈ -3.20`, and `nodematch(primary_genre) ≈ 1.27` (z≈31). These confirm the sparse baseline plus strong closure/degree structure, and isolate genre homophily as the similarity effect worth carrying forward.

## Model 3 — Control for Opportunity/Exposure
Uses the simplified Model 2 structure plus time/productivity controls, estimated via Contrastive Divergence for stability.

In [ ]:
m3_terms <- m2_terms
if (has_vertex_attr("tenure_std")) {
  m3_terms <- c(m3_terms, 'absdiff("tenure_std")')
}
exposure_covs <- c("productivity_std","collab_count_std","popularity_std","followers_std")
for (attr in exposure_covs) {
  if (has_vertex_attr(attr)) {
    m3_terms <- c(m3_terms, sprintf('nodecov("%s")', attr))
  }
}
m3_formula <- ergm_formula_from_terms(m3_terms)
m3_fast <- ergm(m3_formula, estimate = "CD")
summary(m3_fast)

Starting contrastive divergence estimation via CD-MCMLE:

Iteration 1 of at most 60:

Convergence test P-value:0e+00

1 
The log-likelihood improved by 1.479.

Iteration 2 of at most 60:

Convergence test P-value:4.9e-323

1 
The log-likelihood improved by 1.552.

Iteration 3 of at most 60:

Convergence test P-value:1.1e-297

1 
The log-likelihood improved by 1.414.

Iteration 4 of at most 60:

Convergence test P-value:4.7e-287

1 
The log-likelihood improved by 1.421.

Iteration 5 of at most 60:

Convergence test P-value:3.5e-213

1 
The log-likelihood improved by 1.282.

Iteration 6 of at most 60:

Convergence test P-value:8.2e-44

1 
The log-likelihood improved by 0.2012.

Iteration 7 of at most 60:

Convergence test P-value:4e-07

1 
The log-likelihood improved by 0.02739.

Iteration 8 of at most 60:

Convergence test P-value:6.5e-02

1 
The log-likelihood improved by 0.007712.

Iteration 9 of at most 60:

Convergence test P-value:9.2e-01

Convergence detected. Stopping.

1 
The lo

Call:
ergm(formula = net ~ edges + gwesp(0.5, fixed = TRUE) + gwdegree(0.8, 
    fixed = TRUE) + nodematch("primary_genre") + absdiff("time_std") + 
    nodecov("num_songs_std"), estimate = "CD")

Contrastive Divergence Results:

                        Estimate Std. Error MCMC % z value Pr(>|z|)
edges                   -11.5381         NA     NA      NA       NA
gwesp.fixed.0.5           5.1175         NA     NA      NA       NA
gwdeg.fixed.0.8           1.4596         NA     NA      NA       NA
nodematch.primary_genre   1.7239         NA     NA      NA       NA
absdiff.time_std          0.1707         NA     NA      NA       NA
nodecov.num_songs_std     0.1846         NA     NA      NA       NA


### Model 3 results
The CD fit keeps structural terms strong (`gwesp ≈ 5.30`, `gwdegree ≈ 1.29`) while showing genre homophily (`≈ 2.02`), productivity (`nodecov(num_songs_std) ≈ 0.17`), and tenure differences (`absdiff(time_std) ≈ 0.15`) all boost tie odds. With successful CD convergence, we can now run diagnostics/GOF for this exposure-controlled specification.

## Model 4 — Add Weak-Tie Edge Covariate
Builds directly on the stabilized Model 3 terms and fits via CD so we can safely test the low-overlap weak-tie covariate.

In [ ]:
m4_terms <- m3_terms
edgecov_candidates <- c("recency_weight_mat","weight_size_adj_mat","same_country_mat",
                        "same_region_city_mat","same_primary_label_mat","roles_overlap_mat",
                        "genres_overlap_mat","labels_overlap_mat","mean_team_size_cowrite_joint_mat")
for (varname in edgecov_candidates) {
  if (exists(varname) && !is.null(get(varname))) {
    m4_terms <- c(m4_terms, sprintf("edgecov(%s)", varname))
  }
}
m4_formula <- ergm_formula_from_terms(m4_terms)
m4_fast <- ergm(m4_formula, estimate = "CD")
summary(m4_fast)

Starting contrastive divergence estimation via CD-MCMLE:

Iteration 1 of at most 60:

Convergence test P-value:0e+00

1 
The log-likelihood improved by 2.265.

Iteration 2 of at most 60:

Convergence test P-value:0e+00

1 
The log-likelihood improved by 1.608.

Iteration 3 of at most 60:

Convergence test P-value:0e+00

1 
The log-likelihood improved by 1.23.

Iteration 4 of at most 60:

Convergence test P-value:0e+00

1 
The log-likelihood improved by 1.295.

Iteration 5 of at most 60:

Convergence test P-value:1.3e-263

1 
The log-likelihood improved by 1.224.

Iteration 6 of at most 60:

Convergence test P-value:1.2e-58

1 
The log-likelihood improved by 0.2312.

Iteration 7 of at most 60:

Convergence test P-value:8.2e-08

1 
The log-likelihood improved by 0.02724.

Iteration 8 of at most 60:

Convergence test P-value:8.1e-03

1 
The log-likelihood improved by 0.009864.

Iteration 9 of at most 60:

Convergence test P-value:1.3e-01

1 
The log-likelihood improved by 0.005521.

Itera

Call:
ergm(formula = net ~ edges + gwesp(0.5, fixed = TRUE) + gwdegree(0.8, 
    fixed = TRUE) + nodematch("primary_genre") + absdiff("time_std") + 
    nodecov("num_songs_std") + edgecov(low_overlap_mat), estimate = "CD")

Contrastive Divergence Results:

                         Estimate Std. Error MCMC % z value Pr(>|z|)
edges                   -12.43081         NA     NA      NA       NA
gwesp.fixed.0.5           5.18472         NA     NA      NA       NA
gwdeg.fixed.0.8           0.18825         NA     NA      NA       NA
nodematch.primary_genre   2.73406         NA     NA      NA       NA
absdiff.time_std          0.03715         NA     NA      NA       NA
nodecov.num_songs_std     0.05114         NA     NA      NA       NA
edgecov.low_overlap_mat  93.67431         NA     NA      NA       NA


### Model 4 results
CD converged, but `edgecov(low_overlap)` produced an extreme estimate (~98) with a non-varying-statistic warning, so we can’t yet interpret the weak-tie coefficient. Other parameters stay in line (`gwesp ≈ 5.13`, `nodematch ≈ 2.72`, `gwdegree ≈ 0.43`), suggesting we may need to rescale or sparse-threshold the low-overlap matrix before retesting this mechanism.

In [ ]:
g_names   <- igraph::V(g)$name
deg_obs   <- igraph::degree(g, mode = "all")
tri_obs   <- igraph::count_triangles(g, vids = igraph::V(g))

In [ ]:
nsim <- 100
sims <- simulate(
  m1_fast,
  nsim      = nsim,
  output    = "network",
  control   = control.simulate.ergm(
    MCMC.burnin  = 1e5,
    MCMC.interval= 1e3
  )
)

In [ ]:
deg_acc <- numeric(length(g_names))
tri_acc <- numeric(length(g_names))

for (s in seq_len(nsim)) {
  net_s <- sims[[s]]
  g_s   <- asIgraph(net_s)
  g_s   <- igraph::simplify(g_s, remove.multiple = TRUE, remove.loops = TRUE)

  # Ensure vertex order aligns by name
  # (asIgraph should preserve order, but we make it explicit & robust)
  sim_names <- igraph::V(g_s)$name
  m <- match(g_names, sim_names)

  # Some simulations might drop isolated vertices if names are missing; guard that:
  # (If any NA in m, replace with 0-length or 0 stats.)
  d_s <- numeric(length(g_names))
  t_s <- numeric(length(g_names))

  d_s[!is.na(m)] <- igraph::degree(g_s, mode = "all")[m[!is.na(m)]]
  t_s[!is.na(m)] <- igraph::count_triangles(g_s, vids = igraph::V(g_s))[m[!is.na(m)]]

  deg_acc <- deg_acc + d_s
  tri_acc <- tri_acc + t_s
}

In [ ]:
deg_exp <- deg_acc / nsim
tri_exp <- tri_acc / nsim

In [ ]:
node_summary <- data.frame(
  node            = g_names,
  degree_obs      = as.numeric(deg_obs),
  degree_exp      = as.numeric(deg_exp),
  tri_obs         = as.numeric(tri_obs),
  tri_exp         = as.numeric(tri_exp),
  residual_degree = as.numeric(deg_obs - deg_exp),
  residual_tri    = as.numeric(tri_obs - tri_exp),
  stringsAsFactors = FALSE
)

In [ ]:
head(node_summary)

,node,degree_obs,degree_exp,tri_obs,tri_exp,residual_degree,residual_tri
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,777a21a8-0d0f-4cf3-86b4-65bc0eba5649,2,0,1,0,2,1
2,703c557f-82bb-4646-ae2f-b5a3ef3f6148,12,0,66,0,12,66
3,25848dee-8562-4a78-b375-3a80b61da629,5,0,10,0,5,10
4,3e7bcc53-53d4-41d8-afbc-69f82b858fd9,6,0,15,0,6,15
5,43a0ce3a-a3d5-45c0-b944-bd8f5e021237,5,0,4,0,5,4
6,89aa5ecb-59ad-46f5-b3eb-2d424e941f19,23,0,56,0,23,56


In [ ]:
# https://chatgpt.com/share/e/6913d3f1-55f8-800a-8685-df369efa7346
fast_node_gwesp_score <- function(g, decay = 0.5) {
  gi <- g
  # shared partners per edge
  el <- as_edgelist(gi, names = FALSE)
  A <- as_adj(gi, sparse = TRUE)
  sp <- function(i, j) sum(A[i, ] & A[j, ])  # shared partners count

  # geometric weight with diminishing returns
  # NOTE: This mirrors the idea of GWESP weighting (more SP -> higher, but with diminishing gains).
  # You don't need the exact internal ERGM transform to get a useful ranking score.
  w_from_s <- function(s, decay) 1 - exp(-decay * s)

  edge_w <- numeric(nrow(el))
  for (e in seq_len(nrow(el))) {
    s_ij <- sp(el[e, 1], el[e, 2])
    edge_w[e] <- w_from_s(s_ij, decay)
  }

  n <- vcount(gi)
  node_w <- numeric(n)
  for (e in seq_len(nrow(el))) {
    i <- el[e, 1]; j <- el[e, 2]
    node_w[i] <- node_w[i] + edge_w[e]
    node_w[j] <- node_w[j] + edge_w[e]
  }

  data.frame(
    node = V(gi)$name %||% as.character(seq_len(n)),
    gwesp_like_node_score = node_w,
    stringsAsFactors = FALSE
  )
}
gwesp_contribs <- fast_node_gwesp_score(g)

Warning message:
“`as_adj()` was deprecated in igraph 2.1.0.
ℹ Please use `as_adjacency_matrix()` instead.”


In [ ]:
head(gwesp_contribs)

,node,gwesp_like_node_score
,<chr>,<dbl>
1,777a21a8-0d0f-4cf3-86b4-65bc0eba5649,0.7869387
2,703c557f-82bb-4646-ae2f-b5a3ef3f6148,11.9509587
3,25848dee-8562-4a78-b375-3a80b61da629,4.3233236
4,3e7bcc53-53d4-41d8-afbc-69f82b858fd9,5.5074900
5,43a0ce3a-a3d5-45c0-b944-bd8f5e021237,2.6833004
6,89aa5ecb-59ad-46f5-b3eb-2d424e941f19,19.1703870


# Final frame creation
figured out how to get most features, so now we create

In [ ]:
clust_local <- transitivity(g, type = "local", isolates = "zero") # clustering

In [ ]:
triangles_per_node <- count_triangles(g)  # triange count

In [ ]:
deg <- degree(g, mode = "all")
open_wedges <- choose(deg, 2) - triangles_per_node
open_wedges[deg < 2] <- 0 # Open wedges (two-paths that are not closed) (per node)

In [ ]:
gwesp_node_score <- function(gi, decay = 0.5) {
  A  <- as_adj(gi, sparse = TRUE)           # 0/1 adjacency
  SP <- A %*% A                              # shared partners between i and j
  el <- as_edgelist(gi, names = FALSE)

  # geometric diminishing-returns weight aligned with gwesp’s spirit
  w_from_s <- function(s) 1 - exp(-decay * s)

  edge_w <- numeric(nrow(el))
  for (e in seq_len(nrow(el))) {
    s_ij <- SP[el[e,1], el[e,2]]
    edge_w[e] <- w_from_s(s_ij)
  }
  node_w <- numeric(vcount(gi))
  for (e in seq_len(nrow(el))) {
    i <- el[e,1]; j <- el[e,2]
    node_w[i] <- node_w[i] + edge_w[e]
    node_w[j] <- node_w[j] + edge_w[e]
  }
  node_w
}

gwesp_score <- gwesp_node_score(g, decay = 0.5)

In [ ]:
deg <- degree(g, mode = "all")

In [ ]:
eig_cen  <- eigen_centrality(g)$vector
pagerank <- page_rank(g)$vector
kcore    <- coreness(g)
betw     <- betweenness(g, directed = is.directed(g), normalized = TRUE)
close    <- closeness(g, normalized = TRUE)

Warning message:
“`is.directed()` was deprecated in igraph 2.0.0.
ℹ Please use `is_directed()` instead.”


In [ ]:
same_genre_share <- rep(NA_real_, vcount(g))
if (!is.null(igraph::V(g)$primary_genre)) {
  pg <- igraph::V(g)$primary_genre
  for (i in seq_len(vcount(g))) {
    nbrs <- igraph::neighbors(g, i, mode = "all")
    if (length(nbrs) == 0) {
      next
    }
    vals <- pg[nbrs]
    same_genre_share[i] <- mean(vals == pg[i], na.rm = TRUE)
  }
}


In [ ]:
gi <- g

close[!is.finite(close)] <- 0

node_ids <- if (!is.null(igraph::V(gi)$name)) igraph::V(gi)$name else as.character(seq_len(igraph::vcount(gi)))

gwesp_lookup <- stats::setNames(gwesp_contribs$gwesp_like_node_score, gwesp_contribs$node)
gwesp_vec <- as.numeric(gwesp_lookup[node_ids])
gwesp_vec[is.na(gwesp_vec)] <- 0

node_core <- data.frame(
  mbid              = node_ids,
  clustering_local  = as.numeric(clust_local),
  triangles         = as.numeric(triangles_per_node),
  open_wedges       = as.numeric(open_wedges),
  gwesp_contrib     = gwesp_vec,
  degree            = as.numeric(deg),
  eigencentrality   = as.numeric(eig_cen),
  pagerank          = as.numeric(pagerank),
  kcore             = as.numeric(kcore),
  betweenness       = as.numeric(betw),
  closeness         = as.numeric(close),
  same_genre_share  = as.numeric(same_genre_share),
  stringsAsFactors  = FALSE
)

node_attr_cols <- c(
  "mbid","artist_name","window_years","years_active","releases_total","releases_per_year",
  "tracks_total","productivity_total","productivity_std","collab_count","collab_count_std",
  "tenure_years","tenure_std","recency_index","recency_index_std","popularity","popularity_std",
  "followers","followers_std","unique_collaborator_count","collab_track_rate","label_diversity_count",
  "label_churn","label_hhi","genre_count","genre_entropy","artist_country","artist_region_city",
  "primary_genre","primary_role","primary_label","all_genres_str","all_roles_str","all_labels_str",
  "location_country_known","location_region_city_known","missing_recordings_flag",
  "missing_genres_flag","missing_labels_flag","missing_releases_flag","recency_index_missing_flag"
)
node_attrs <- dplyr::select(nodes_df, dplyr::any_of(node_attr_cols))

node_features <- node_core %>%
  dplyr::left_join(node_summary, by = c("mbid" = "node")) %>%
  dplyr::left_join(node_attrs, by = "mbid")

In [ ]:
head(node_features)

,mbid,clustering_local,triangles,open_wedges,gwesp_contrib,degree,eigencentrality,pagerank,kcore,betweenness,closeness,same_genre_share
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,777a21a8-0d0f-4cf3-86b4-65bc0eba5649,1.0000000,1,0,0.7869387,2,2.509173e-05,0.0001814516,2,0.000000000,0.1492665,1.0000000
2,703c557f-82bb-4646-ae2f-b5a3ef3f6148,1.0000000,66,0,11.9509587,12,1.472325e-04,0.0004127831,12,0.000000000,0.1707496,0.9166667
3,25848dee-8562-4a78-b375-3a80b61da629,1.0000000,10,0,4.3233236,5,0.000000e+00,0.0002406972,5,0.000000000,0.4303797,1.0000000
4,3e7bcc53-53d4-41d8-afbc-69f82b858fd9,1.0000000,15,0,5.5074900,6,5.511010e-03,0.0002849998,6,0.000000000,0.2302610,0.5000000
5,43a0ce3a-a3d5-45c0-b944-bd8f5e021237,0.4000000,4,6,2.6833004,5,5.220216e-06,0.0003960507,3,0.002117678,0.1470930,0.4000000
6,89aa5ecb-59ad-46f5-b3eb-2d424e941f19,0.2213439,56,197,19.1703870,23,1.426881e-02,0.0011203706,9,0.004394862,0.2373170,0.1304348


In [ ]:
write.csv(node_features, "node_features.csv", row.names = FALSE)